# Khmer Sign Language — Train on Google Colab (PyTorch)

Trains and compares the three approaches on a Colab GPU: **LSTM**, **GRU**, **Transformer encoder** — all trained from scratch on the identical data split.

**Before running this notebook:**
1. On your own machine, run `collect_data.py` then `extract_landmarks.py` (these need your webcam, so they stay local). This produces `data/X.npy`, `data/y.npy`, `data/labels.npy`.
2. Upload those 3 files to Google Drive, e.g. into `MyDrive/KhmerSignLanguage/data/`.
3. Also upload `src/utils.py`, `src/dataset.py`, and the `src/models/` folder to `MyDrive/KhmerSignLanguage/src/` so this notebook can import them.
4. **Runtime -> Change runtime type -> GPU (T4)**.
5. Run the cells below in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/KhmerSignLanguage'
DATA_DIR    = f'{PROJECT_DIR}/data'
MODELS_DIR  = f'{PROJECT_DIR}/models'
RESULTS_DIR = f'{PROJECT_DIR}/results'
SRC_DIR     = f'{PROJECT_DIR}/src'

import os, sys
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

import torch
print('Torch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Stage 3 — Preprocess (fixed 70/15/15 train/val/test split)

Same split recipe as `src/preprocess.py`, saved once so every approach below trains and is scored on identical data.

In [ ]:
import json
import numpy as np
from sklearn.model_selection import train_test_split

from utils import set_seed

SEED = 42
VAL_SIZE = 0.15
TEST_SIZE = 0.15
set_seed(SEED)

X = np.load(os.path.join(DATA_DIR, 'X.npy'))
y = np.load(os.path.join(DATA_DIR, 'y.npy'))
labels = np.load(os.path.join(DATA_DIR, 'labels.npy'), allow_pickle=True)
num_classes = len(labels)
print(f'Loaded X {X.shape}, y {y.shape}, classes: {list(labels)}')

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
val_fraction = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_fraction, random_state=SEED, stratify=y_temp)

print(f'Train {X_train.shape[0]} | Val {X_val.shape[0]} | Test {X_test.shape[0]}')

for name, arr in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
                   ('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    np.save(os.path.join(DATA_DIR, f'{name}.npy'), arr)

input_size = X_train.shape[-1]

## Stage 4 — Train all three approaches

Uses the exact same `LandmarkSequenceDataset` and model classes as the local `src/` code, so results are directly comparable.

In [ ]:
import time
import torch.nn as nn
from torch.utils.data import DataLoader

from dataset import LandmarkSequenceDataset
from models import build_model
from utils import get_device, device_name, count_trainable_params, save_checkpoint

device = get_device()
EPOCHS = 200
BATCH_SIZE = 16
PATIENCE = 25
LR = 1e-3
WEIGHT_DECAY = 0.0

def train_one(model_name):
    set_seed(SEED)
    run_dir = os.path.join(RESULTS_DIR, model_name)
    os.makedirs(run_dir, exist_ok=True)

    train_loader = DataLoader(LandmarkSequenceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(LandmarkSequenceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

    model = build_model(model_name, input_size=input_size, num_classes=num_classes).to(device)
    n_params = count_trainable_params(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, epochs_without_improvement = 0.0, 0
    checkpoint_path = os.path.join(run_dir, 'checkpoint.pt')
    best_model_path = os.path.join(MODELS_DIR, f'{model_name}_khmer_sign.pt')
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                v_loss += criterion(logits, yb).item() * xb.size(0)
                v_correct += (logits.argmax(dim=1) == yb).sum().item()
                v_total += xb.size(0)
        val_loss, val_acc = v_loss / v_total, v_correct / v_total
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            save_checkpoint(checkpoint_path, model, optimizer, epoch, best_val_acc)
            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_without_improvement += 1
        if epoch % 10 == 0 or epochs_without_improvement == 0:
            print(f'[{model_name}] epoch {epoch}: train_acc {train_acc:.3f} val_acc {val_acc:.3f}')
        if epochs_without_improvement >= PATIENCE:
            print(f'[{model_name}] early stopping at epoch {epoch}')
            break

    training_time = time.time() - start
    with open(os.path.join(run_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    with open(os.path.join(run_dir, 'meta.json'), 'w') as f:
        json.dump({
            'model': model_name, 'trainable_params': n_params,
            'training_time_seconds': round(training_time, 1),
            'device': device_name(device), 'epochs_run': len(history['train_loss']),
            'best_val_acc': best_val_acc,
            'hyperparameters': {'lr': LR, 'weight_decay': WEIGHT_DECAY, 'batch_size': BATCH_SIZE, 'patience': PATIENCE, 'seed': SEED},
        }, f, indent=2)
    print(f'[{model_name}] done. best val_acc={best_val_acc:.3f}, params={n_params:,}, time={training_time:.1f}s')

for m in ['lstm', 'gru', 'transformer']:
    train_one(m)

## Stage 5 — Evaluate all three on the SAME held-out test set

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate_one(model_name):
    run_dir = os.path.join(RESULTS_DIR, model_name)
    model_path = os.path.join(MODELS_DIR, f'{model_name}_khmer_sign.pt')
    model = build_model(model_name, input_size=input_size, num_classes=num_classes).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    loader = DataLoader(LandmarkSequenceDataset(X_test, y_test), batch_size=32, shuffle=False)
    preds, trues, probs_all = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            probs = torch.softmax(model(xb), dim=1)
            preds.append(probs.argmax(dim=1).cpu().numpy())
            probs_all.append(probs.cpu().numpy())
            trues.append(yb.numpy())
    y_pred, y_true, y_prob = np.concatenate(preds), np.concatenate(trues), np.concatenate(probs_all)

    acc = float(np.mean(y_pred == y_true))
    report = classification_report(y_true, y_pred, target_names=labels, output_dict=True, zero_division=0)
    print(f'\n=== {model_name.upper()} test accuracy: {acc*100:.2f}% ===')
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

    metrics = {
        'model': model_name, 'test_accuracy': acc,
        'precision_macro': report['macro avg']['precision'],
        'recall_macro': report['macro avg']['recall'],
        'f1_macro': report['macro avg']['f1-score'],
        'precision_weighted': report['weighted avg']['precision'],
        'recall_weighted': report['weighted avg']['recall'],
        'f1_weighted': report['weighted avg']['f1-score'],
        'per_class': {cls: report[cls] for cls in labels},
    }
    with open(os.path.join(run_dir, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, xticks_rotation=45)
    ax.set_title(f'Confusion Matrix - {model_name.upper()}')
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'confusion_matrix.png'), dpi=150)
    plt.show()

    wrong_idx = np.where(y_pred != y_true)[0]
    pd.DataFrame([{'test_index': int(i), 'true_label': labels[y_true[i]],
                    'predicted_label': labels[y_pred[i]],
                    'predicted_confidence': float(y_prob[i, y_pred[i]])} for i in wrong_idx]
                 ).to_csv(os.path.join(run_dir, 'errors.csv'), index=False)
    return metrics

all_metrics = {m: evaluate_one(m) for m in ['lstm', 'gru', 'transformer']}

## Stage 6 — Comparison table and figures

In [ ]:
rows = []
histories = {}
for m in ['lstm', 'gru', 'transformer']:
    run_dir = os.path.join(RESULTS_DIR, m)
    with open(os.path.join(run_dir, 'meta.json')) as f:
        meta = json.load(f)
    with open(os.path.join(run_dir, 'metrics.json')) as f:
        metrics = json.load(f)
    with open(os.path.join(run_dir, 'history.json')) as f:
        histories[m] = json.load(f)
    rows.append({'approach': m.upper(), 'test_accuracy': metrics['test_accuracy'],
                 'precision_macro': metrics['precision_macro'], 'recall_macro': metrics['recall_macro'],
                 'f1_macro': metrics['f1_macro'], 'trainable_params': meta['trainable_params'],
                 'training_time_s': meta['training_time_seconds'], 'epochs_run': meta['epochs_run'],
                 'device': meta['device']})

df = pd.DataFrame(rows).sort_values('test_accuracy', ascending=False)
df.to_csv(os.path.join(RESULTS_DIR, 'comparison_table.csv'), index=False)
with open(os.path.join(RESULTS_DIR, 'comparison_table.md'), 'w') as f:
    f.write(df.to_markdown(index=False))
display(df)

fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(df)); width = 0.35
ax.bar([i - width/2 for i in x], df['test_accuracy'], width, label='Test Accuracy', color='steelblue')
ax.bar([i + width/2 for i in x], df['f1_macro'], width, label='Macro F1', color='tomato')
ax.set_xticks(list(x)); ax.set_xticklabels(df['approach']); ax.set_ylim(0, 1.0)
ax.set_title('Test Accuracy & F1 by Approach'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(os.path.join(fig_dir, 'accuracy_comparison.png'), dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for m, h in histories.items():
    ax.plot(h['val_acc'], label=f'{m.upper()} val_acc')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy')
ax.set_title('Validation Accuracy Learning Curves'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(fig_dir, 'learning_curves.png'), dpi=150); plt.show()

## Stage 7 — Hyperparameter tuning

Tries 3 learning rates x 2 weight-decay values (6 combinations) for GRU (the best approach) and LSTM. Combinations are ranked by **validation** accuracy, never the test set. Saves `results/<model>/hparam_search.json`.

In [ ]:
import itertools
import pandas as pd

LEARNING_RATES = [1e-3, 5e-4, 1e-4]
WEIGHT_DECAYS  = [0.0, 1e-4]      # 0.0 = off, 1e-4 = L2 regularization on
SEARCH_EPOCHS  = 60

def run_search(model_name):
    train_loader = DataLoader(LandmarkSequenceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(LandmarkSequenceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    criterion = nn.CrossEntropyLoss()
    rows = []

    for lr, wd in itertools.product(LEARNING_RATES, WEIGHT_DECAYS):
        set_seed(SEED)
        model = build_model(model_name, input_size=input_size, num_classes=num_classes).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        best_val_acc, min_val_loss = 0.0, float('inf')
        start = time.time()

        for epoch in range(SEARCH_EPOCHS):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                criterion(model(xb), yb).backward()
                optimizer.step()

            model.eval()
            v_loss, v_correct, v_total = 0.0, 0, 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    logits = model(xb)
                    v_loss += criterion(logits, yb).item() * xb.size(0)
                    v_correct += (logits.argmax(dim=1) == yb).sum().item()
                    v_total += xb.size(0)
            best_val_acc = max(best_val_acc, v_correct / v_total)
            min_val_loss = min(min_val_loss, v_loss / v_total)

        rows.append({'lr': lr, 'weight_decay': wd,
                     'best_val_acc': round(best_val_acc, 4),
                     'min_val_loss': round(min_val_loss, 4),
                     'seconds': round(time.time() - start, 1)})
        print(f'[{model_name}] lr={lr} weight_decay={wd} -> best val_acc {best_val_acc:.3f}')

    # Choose by VALIDATION accuracy (ties broken by lower validation loss) - never the test set
    df = pd.DataFrame(rows).sort_values(['best_val_acc', 'min_val_loss'], ascending=[False, True]).reset_index(drop=True)
    run_dir = os.path.join(RESULTS_DIR, model_name)
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, 'hparam_search.json'), 'w') as f:
        json.dump({'model': model_name, 'search_epochs': SEARCH_EPOCHS,
                   'results': df.to_dict('records'), 'best': df.iloc[0].to_dict()}, f, indent=2)
    print(f'[{model_name}] BEST: lr={df.iloc[0]["lr"]}, weight_decay={df.iloc[0]["weight_decay"]}')
    return df

for m in ['gru', 'lstm']:
    display(run_search(m))


## Next step

Everything under `MyDrive/KhmerSignLanguage/{models,results}` mirrors the local `models/` and `results/` folders. Download it back to your local repo (or keep model weights linked from Drive in the README if they're over 50MB — see the submission checklist), then run `src/hyperparam_search.py --model <best approach>` locally or in Colab to document the tuning step.